# Domain D: Time Series Forecasting - Exploration

This notebook explores the 10 approaches for time series forecasting.

## Setup

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.domain_d_time_series.data_generator import create_timeseries_dataset
from src.domain_d_time_series.approach_01_statistical import ARIMAForecaster
from src.domain_d_time_series.approach_03_tree_based import TreeBasedForecaster
from src.domain_d_time_series.approach_04_rnn_lstm import LSTMForecaster
from src.core.metrics import compute_timeseries_metrics

## Generate Dataset

In [ ]:
dataset = create_timeseries_dataset(
    n_samples=2000,
    lookback=48,
    forecast_horizon=12
)

print(f"Train X shape: {dataset['train']['X'].shape}")
print(f"Train y shape: {dataset['train']['y'].shape}")
print(f"Test X shape: {dataset['test']['X'].shape}")
print(f"Lookback: {dataset['lookback']}")
print(f"Forecast horizon: {dataset['forecast_horizon']}")

## Visualize Time Series

In [ ]:
raw_series = dataset['raw_series'][:500, 0]

plt.figure(figsize=(14, 4))
plt.plot(raw_series)
plt.xlabel('Time')
plt.ylabel('Value')
plt.title('Sample Time Series')
plt.show()

## Compare Approaches

In [ ]:
results = []
forecast_horizon = dataset['forecast_horizon']

# Naive baseline
naive_pred = np.tile(dataset['test']['X'][:, -1:, :], (1, forecast_horizon, 1))

# ARIMA
arima_model = ARIMAForecaster({'order': (2, 1, 2)})
arima_model.train(
    dataset['train']['X'], dataset['train']['y'],
    train_series=dataset['train_series'],
    forecast_horizon=forecast_horizon
)
arima_pred = arima_model.predict(dataset['test']['X'])
arima_metrics = compute_timeseries_metrics(
    dataset['test']['y'].reshape(-1),
    arima_pred.reshape(-1),
    naive_pred.reshape(-1)
)
results.append({'Approach': 'ARIMA', **arima_metrics})

# Tree-Based
tree_model = TreeBasedForecaster({'n_estimators': 50})
tree_model.train(
    dataset['train']['X'], dataset['train']['y'],
    forecast_horizon=forecast_horizon
)
tree_pred = tree_model.predict(dataset['test']['X'])
tree_metrics = compute_timeseries_metrics(
    dataset['test']['y'].reshape(-1),
    tree_pred.reshape(-1),
    naive_pred.reshape(-1)
)
results.append({'Approach': 'XGBoost', **tree_metrics})

# LSTM
lstm_model = LSTMForecaster({'epochs': 20, 'hidden_dim': 32})
lstm_model.train(
    dataset['train']['X'], dataset['train']['y'],
    forecast_horizon=forecast_horizon
)
lstm_pred = lstm_model.predict(dataset['test']['X'])
lstm_metrics = compute_timeseries_metrics(
    dataset['test']['y'].reshape(-1),
    lstm_pred.reshape(-1),
    naive_pred.reshape(-1)
)
results.append({'Approach': 'LSTM', **lstm_metrics})

# Display results
df = pd.DataFrame(results)
df[['Approach', 'mae', 'rmse', 'mape', 'mase']]

## Visualize Predictions

In [ ]:
sample_idx = 0

plt.figure(figsize=(12, 5))
plt.plot(range(forecast_horizon), dataset['test']['y'][sample_idx, :, 0], 
         'b-', label='Actual', linewidth=2)
plt.plot(range(forecast_horizon), tree_pred[sample_idx, :, 0], 
         'r--', label='XGBoost', linewidth=2)
plt.plot(range(forecast_horizon), lstm_pred[sample_idx, :, 0], 
         'g--', label='LSTM', linewidth=2)
plt.xlabel('Forecast Horizon')
plt.ylabel('Value')
plt.title('Forecast Comparison')
plt.legend()
plt.show()

## Philosophy Comparison

In [ ]:
for model in [arima_model, tree_model, lstm_model]:
    philosophy = model.get_philosophy()
    print(f"\n{model.name}:")
    print(f"  Mental Model: {philosophy['mental_model']}")
    print(f"  Strengths: {philosophy['strengths']}")
    print(f"  Best For: {philosophy['best_for']}")